# 07 — Telephony — Telnyx

Call Control, Ed25519, AMD, DTMF, track filter, anti-replay.


## Optional: run in Docker

Uncomment the cell below to launch JupyterLab + EmbeddedServer in a container. It's idempotent and a no-op once you're already inside the container. See `examples/notebooks/python/Dockerfile` and `docker-compose.yml` for what it builds.


In [ ]:
# Optional — launch the patter-notebooks Docker stack from this cell.
# Skip this cell to run on your host Python. Idempotent if uncommented.
#
# import _setup
# _setup.start_docker()             # build + up -d, prints http://localhost:8888
# _setup.start_docker(open_url=True)  # …and open the browser tab


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | provider keys auto-detected |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + carrier creds |


In [ ]:
%load_ext autoreload
%autoreload 2

import _setup
env = _setup.load()
print(f'getpatter version target: {env.patter_version}')


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline (no network, no carrier calls).


In [ ]:
import sys
import getpatter
with _setup.cell('version_check', tier=1, env=env) as ok:
    if ok:
        print(f'getpatter {getpatter.__version__} on Python {sys.version.split()[0]}')
        assert getpatter.__version__ >= env.patter_version, \
            f'installed {getpatter.__version__} < target {env.patter_version}'


### Local mode
Construct a Patter instance with a Twilio carrier. No API key — runs entirely on your machine.


In [ ]:
from getpatter import Patter, Twilio
with _setup.cell('local_mode', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(
                account_sid='ACtest00000000000000000000000000',
                auth_token='test',
            ),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        # Verify carrier is wired up via LocalConfig.
        assert p._local_config.telephony_provider == 'twilio'
        assert p._local_config.phone_number == '+15555550100'
        print(f'provider={p._local_config.telephony_provider}  phone={p._local_config.phone_number}')


### Cloud mode (coming soon)
When `api_key=` is supported, Patter cloud handles telephony. For now, the SDK raises `NotImplementedError` — this cell verifies the guard.


In [ ]:
from getpatter import Patter
with _setup.cell('cloud_mode', tier=1, env=env) as ok:
    if ok:
        try:
            Patter(api_key='pt_test_xxx')
            raise AssertionError('expected NotImplementedError — cloud mode guard missing')
        except NotImplementedError as exc:
            print(f'cloud mode guard OK: {exc}')


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline* (STT + LLM + TTS). The factory derives the engine from `engine=` / `stt=`/`tts=`.


In [ ]:
from getpatter import Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI
with _setup.cell('agent_engines', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(account_sid='ACtest00000000000000000000000000',
                           auth_token='test'),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        rt = p.agent(system_prompt='hi', engine=OpenAIRealtime(api_key='sk-test'))
        cv = p.agent(system_prompt='hi', engine=ElevenLabsConvAI(api_key='el-test', agent_id='a1'))
        pl = p.agent(system_prompt='hi')  # default: OpenAI Realtime fallback
        assert rt.provider == 'openai_realtime', rt.provider
        assert cv.provider == 'elevenlabs_convai', cv.provider
        assert pl.provider == 'openai_realtime', pl.provider
        print(f'rt.provider={rt.provider}  cv.provider={cv.provider}  pl.provider={pl.provider}')


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline (no network, no carrier calls).


### Local mode
Construct a Patter instance with a Twilio carrier. No API key — runs entirely on your machine.


### Cloud mode
Same SDK, just an `api_key=` instead of a carrier — Patter cloud handles telephony.


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline* (STT + LLM + TTS). The factory derives the mode from `engine=` / `stt=`/`tts=`.


## §2: Feature Tour

One cell per feature/provider. Missing keys auto-skip.


## §2 — Feature Tour

Exercises Telnyx carrier construction and Ed25519 webhook signature verification.


### Telnyx carrier construction


In [ ]:
from getpatter import Patter, Telnyx
with _setup.cell('telnyx_carrier', tier=1, env=env) as ok:
    if ok:
        carrier = Telnyx(
            api_key='KEY0_TEST_TELNYX',
            public_key='',  # Ed25519 public key (omitted for offline test)
        )
        p = Patter(
            carrier=carrier,
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        lc = p._local_config
        print(f'carrier:  {lc.telephony_provider}')
        print(f'phone:    {lc.phone_number}')
        assert lc.telephony_provider == 'telnyx'


### Ed25519 sign + verify


In [ ]:
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey, Ed25519PublicKey
from cryptography.hazmat.primitives.serialization import (
    Encoding, PrivateFormat, PublicFormat, NoEncryption,
    load_pem_private_key, load_pem_public_key,
)
with _setup.cell('ed25519_verify', tier=1, env=env) as ok:
    if ok:
        # Load test keypair from fixtures.
        priv_pem = _setup.load_fixture('keys/telnyx_test_ed25519_private.pem')
        pub_pem  = _setup.load_fixture('keys/telnyx_test_ed25519_public.pem')
        private_key = load_pem_private_key(priv_pem, password=None)
        public_key  = load_pem_public_key(pub_pem)
        payload = b'telnyx-webhook-payload'
        signature = private_key.sign(payload)
        public_key.verify(signature, payload)  # raises if invalid
        print(f'Ed25519 sign + verify OK  sig_len={len(signature)} bytes')
        # Tampered payload must fail.
        import pytest
        from cryptography.exceptions import InvalidSignature
        try:
            public_key.verify(signature, b'tampered-payload')
            print('ERROR: tampered payload should have raised InvalidSignature')
        except InvalidSignature:
            print('Tampered payload correctly rejected')


### Telnyx anti-replay: timestamp window check


In [ ]:
import time
with _setup.cell('telnyx_timestamp', tier=1, env=env) as ok:
    if ok:
        WINDOW_SECONDS = 300  # ±5 minutes
        now = int(time.time())
        cases = [
            (now,              'fresh   — should pass'),
            (now - 60,         'recent  — should pass'),
            (now - 299,        'edge    — should pass'),
            (now - 301,        'stale   — should reject'),
            (now + 10,         'future  — should pass'),
        ]
        for ts, label in cases:
            age = abs(now - ts)
            accepted = age <= WINDOW_SECONDS
            mark = '✓' if accepted else '✗'
            print(f'  {mark} age={age:3d}s  {label}')


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.


## §3 — Live Appendix

Places a real call through the Telnyx carrier. Requires `ENABLE_LIVE_CALLS=1` and Telnyx credentials.


### Pre-flight checklist


In [ ]:
with _setup.cell('live_preflight', tier=4, required=['TELNYX_API_KEY', 'TELNYX_CONNECTION_ID', 'TELNYX_PHONE_NUMBER', 'TARGET_PHONE_NUMBER'], env=env) as ok:
    if ok:
        print(f'  carrier:       Telnyx {env.telnyx_number}  →  {env.target_number}')
        print(f'  connection_id: {env.telnyx_connection_id}')
        print(f'  webhook:       {env.public_webhook_url or "(ngrok auto-launch)"}')


### Live Telnyx outbound call *(T4)*


In [ ]:
from getpatter import Patter, Telnyx, OpenAIRealtime
with _setup.cell('live_telnyx_call', tier=4, required=['TELNYX_API_KEY', 'TELNYX_CONNECTION_ID', 'TELNYX_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'OPENAI_API_KEY'], env=env) as ok:
    if ok:
        p = Patter(
            carrier=Telnyx(api_key=env.telnyx_key, public_key=env.telnyx_public_key),
            phone_number=env.telnyx_number,
            webhook_url=env.public_webhook_url,
        )
        agent = p.agent(
            system_prompt='You are a Telnyx demo agent. Greet the caller and hang up.',
            engine=OpenAIRealtime(api_key=env.openai_key),
        )
        try:
            await p.call(
                env.target_number,
                agent=agent,
                first_message='Hello from Patter via Telnyx.',
                ring_timeout=env.max_call_seconds,
            )
            print('✓ Telnyx call completed')
        finally:
            _setup.hangup_leftover_calls(env)
